# 28 — Thinking / Reasoning Fine-tune

Teach the ARO model to **reason before it answers**: understand the request,
rephrase it in ARO terms, plan the feature-set structure, and explicitly recall
the rules the 4,000-prompt eval showed it breaks (immutability, prepositions,
built-in verbs, Return/Throw) — *then* generate code.

Training data: `Train/eval_derived/thinking.jsonl` (reasoning traces) +
`eval_feedback_pairs.jsonl`. The notebook evaluates reasoning quality on a
held-out slice **before** and **after** the fine-tune and writes a chart.


In [ ]:
import json, subprocess, sys, tempfile, random, re
from pathlib import Path
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt

REPO = Path.cwd().parents[1] if (Path.cwd().name == "script") else Path("/Users/kris/Projects/ARO/ARO-Lang")
sys.path.insert(0, str(REPO / "Train" / "script"))
import config

# Release chain (folds this booster into the shipped model): anchor on the
# material-boosted model, then the distilled student. Falling through to
# config.BASE_MODEL_ID would train and fuse a 30B mixture-of-experts LoRA into
# what the rest of the chain calls "the student", announced by one print line —
# and NB25 then anchors on this directory, so the release changes architecture
# mid-chain with nothing failing (issue #791).
from fusion_gate import resolve_base, require_fuse_gate
_mdl = Path(config.MODELS_DIR)
BASE_MODEL, _base_reason = resolve_base(
    "thinking",
    [_mdl / "material" / "fused", _mdl / "distill" / "student" / "fused"],
    fallback=config.BASE_MODEL_ID)
print(_base_reason)
FUSED_DIR = _mdl / "thinking" / "fused"
FUSED_DIR.mkdir(parents=True, exist_ok=True)
DATA_IN      = REPO / "Train" / "eval_derived"
WORK         = REPO / "Train" / "data" / "26_thinking"
MLX_DIR      = WORK / "mlx"
ADAPTER_DIR  = WORK / "adapter"
CHART        = REPO / "Train" / "Reports" / "26_thinking_finetune.png"
# Hyper-parameters come from config.HPARAMS['thinking'] (GitLab #795).
# Eight notebooks used to declare these separately and disagree; the
# table in config.py now carries the value AND the reason for it, and
# check_hparams.py fails CI if a literal comes back here.
HP = config.hparams('thinking')
ITERS        = HP['iters']
LORA_LAYERS  = HP['lora_layers']
BATCH_SIZE   = HP['batch_size']
LEARNING_RATE= HP['learning_rate']
for d in (MLX_DIR, ADAPTER_DIR, CHART.parent):
    d.mkdir(parents=True, exist_ok=True)
print("repo:", REPO)

## 1. Load reasoning data → chat messages, split train/valid/held-out

In [ ]:
def load_pairs():
    pairs = []
    for name in ("thinking.jsonl", "eval_feedback_pairs.jsonl"):
        p = DATA_IN / name
        if not p.exists():
            continue
        for line in p.read_text().splitlines():
            if line.strip():
                r = json.loads(line)
                if r.get("instruction") and r.get("output"):
                    pairs.append(r)
    return pairs

pairs = load_pairs()
random.seed(0); random.shuffle(pairs)
holdout = pairs[:120]                 # measure before/after on these
train_p = pairs[120:]
def to_msg(r):
    return {"messages": [
        {"role": "user", "content": r["instruction"]},
        {"role": "assistant", "content": r["output"]},
    ]}
n_val = max(1, len(train_p)//20)
valid = [to_msg(r) for r in train_p[:n_val]]
train = [to_msg(r) for r in train_p[n_val:]]
(MLX_DIR/"train.jsonl").write_text("\n".join(json.dumps(m) for m in train))
(MLX_DIR/"valid.jsonl").write_text("\n".join(json.dumps(m) for m in valid))
print(f"pairs={len(pairs)} train={len(train)} valid={len(valid)} holdout={len(holdout)}")

## 2. Reasoning-quality metrics

For each held-out prompt we score the model's answer on:
* **reasons** — did it emit a non-empty `<think>` block before the code?
* **valid_code** — does the extracted ARO pass `aro check`?


In [ ]:
def has_reasoning(text):
    m = re.search(r"<think>(.*?)</think>", text, re.DOTALL)
    return bool(m and len(m.group(1).strip()) > 40)

def code_valid(text):
    blocks = config.extract_aro_blocks(text)
    if not blocks:
        return False
    wrapped, _ = config.auto_wrap_aro(blocks[0])
    ok, _ = config.aro_check_snippet(wrapped or blocks[0], timeout=20)
    return bool(ok)

def evaluate(gen_fn):
    reasons = valid = 0
    for r in holdout:
        out = gen_fn(r["instruction"])
        reasons += has_reasoning(out)
        valid   += code_valid(out)
    n = len(holdout)
    return {"reasons": 100*reasons/n, "valid_code": 100*valid/n}

## 3. Generation (base model, then adapter). Requires `mlx_lm` + the model.

In [ ]:
from mlx_lm import load as mlx_load, generate as mlx_generate
from mlx_lm.sample_utils import make_sampler

def make_gen(adapter=None):
    model, tok = mlx_load(BASE_MODEL, adapter_path=str(adapter) if adapter else None)
    def gen(instruction):
        msgs = [{"role": "user", "content": instruction}]
        prompt = tok.apply_chat_template(msgs, add_generation_prompt=True, tokenize=False)
        return mlx_generate(model, tok, prompt=prompt, max_tokens=512, verbose=False)
    return gen

print("Evaluating BASE model on held-out ...")
before = evaluate(make_gen(None))
print("before:", before)

## 4. LoRA fine-tune on the reasoning data (mlx-lm, same pattern as NB17)

In [ ]:
cmd = [sys.executable, "-m", "mlx_lm", "lora", "--train",
       "--model", BASE_MODEL,
       "--data", str(MLX_DIR),
       "--adapter-path", str(ADAPTER_DIR),
       "--iters", str(ITERS),
       "--num-layers", str(LORA_LAYERS),
       "--batch-size", str(BATCH_SIZE),
       "--learning-rate", str(LEARNING_RATE)]
print(" ".join(cmd))
subprocess.run(cmd, check=True)

In [ ]:
print("Evaluating FINE-TUNED model on held-out ...")
after = evaluate(make_gen(ADAPTER_DIR))
print("after:", after)

## 5. Chart — reasoning quality before vs after

In [ ]:
metrics = ["reasons", "valid_code"]
labels  = ["% with reasoning", "% valid ARO code"]
b = [before[m] for m in metrics]; a = [after[m] for m in metrics]
x = range(len(metrics)); w = 0.38
fig, ax = plt.subplots(figsize=(8, 5))
ax.bar([i-w/2 for i in x], b, w, label="base", color="#bbbbbb")
ax.bar([i+w/2 for i in x], a, w, label="thinking fine-tune", color="#2ca02c")
for i in x:
    ax.text(i-w/2, b[i]+1, f"{b[i]:.0f}%", ha="center", fontsize=9)
    ax.text(i+w/2, a[i]+1, f"{a[i]:.0f}%", ha="center", fontsize=9)
    ax.text(i, max(b[i],a[i])+6, f"+{a[i]-b[i]:.0f} pts", ha="center", fontsize=9, color="#2ca02c", fontweight="bold")
ax.set_xticks(list(x)); ax.set_xticklabels(labels)
ax.set_ylim(0, 110); ax.set_ylabel("%"); ax.legend()
ax.set_title("ARO Thinking Fine-tune — reasoning quality on held-out prompts")
fig.tight_layout(); fig.savefig(CHART, dpi=120)
print("wrote", CHART)
print("summary:", {"before": before, "after": after})

## Conclusion

`reasons` should rise toward ~100% (the model now emits a planning `<think>`
block) and `valid_code` should climb as the reasoning forces it to respect
immutability, prepositions and the built-in verb set before generating. The
chart is saved to `Train/Reports/thinking_finetune.png` and can be embedded in
the post-training PDF report.


In [ ]:
# --- Gate, then fuse for the release chain -----------------------------------
# `before` and `after` were already measured on the 120-item holdout two cells
# up, charted, and then discarded: a regression fused exactly like an
# improvement. The gate now consumes them (issue #791).
require_fuse_gate("thinking", before, after, n=len(holdout))

# Fuse the thinking adapter into its base so the conversation fine-tune (next)
# and the packager carry the improvement forward. Produces models/thinking/fused.
_fuse = [sys.executable, "-m", "mlx_lm", "fuse",
         "--model", str(BASE_MODEL),
         "--adapter-path", str(ADAPTER_DIR),
         "--save-path", str(FUSED_DIR)]
print(" ".join(_fuse))
subprocess.run(_fuse, check=True)
print("fused thinking model →", FUSED_DIR)

## Record this stage

Every stage records to `Train/experiments.db` under one session id, and
the committed CSV under `Train/runs/<release>/` is exported from it.
Only four stages used to log anything, all four of them training passes —
so the warm start, the distillation, the boosters and packaging left no
record at all (GitLab #812).


In [ ]:
# GitLab #812. Best-effort: bookkeeping must never fail a stage, and
# every name below is optional because stages differ in what they
# produce. Whatever exists is recorded; whatever does not is skipped.
try:
    import sys as _sys_rec
    from pathlib import Path as _Path_rec
    _sys_rec.path.insert(0, str(_Path_rec.cwd()))
    from experiment_db import record_run as _record_run

    _cfg = {}
    _cfg.update(config.hparams_record('thinking'))
    for _name in ('BASE_MODEL', 'ITERS', 'RUN_ITERS', 'n_train',
                  'num_iters', 'train_pairs'):
        _value = globals().get(_name)
        if isinstance(_value, (str, int, float)):
            _cfg[_name] = _value
        elif hasattr(_value, '__len__'):
            _cfg[_name] = len(_value)

    _metrics = {}
    for _name in ('val_losses', 'train_losses'):
        _series = globals().get(_name) or []
        if _series:
            _metrics[f'best_{_name[:-3]}'] = min(_series)
            _metrics[f'final_{_name[:-3]}'] = _series[-1]
    for _name in ('pass_rate', 'reply_rate', 'alignment_rate'):
        if isinstance(globals().get(_name), (int, float)):
            _metrics[_name] = globals()[_name]

    _artifacts = {}
    for _name in ('ADAPTER_DIR', 'FUSED_DIR', 'WARM_ADAPTER',
                  'STUDENT_ADAPTER', 'RELEASE_DIR'):
        if globals().get(_name) is not None:
            _artifacts[_name.lower()] = str(globals()[_name])

    _record_run('NB24', config=_cfg, metrics=_metrics,
                artifacts=_artifacts)
    print('recorded NB24 to experiments.db')
except Exception as _exc:
    print('experiments.db not updated for NB24:', _exc)
